# Exercise 06 : Buyer and Seller

Open `exercises/exercise06-buyer-and-seller.ipynb` and run the setup cell. Build two agents that negotiate a fixed Aurelia filter order under private price policies by replacing the `____` blanks. The solution is `solutions/exercise06-buyer-and-seller.ipynb`.

Worked examples to revisit : module 06 demo06 · conflicts, demo07 · buyer-and-seller.

Made up, not from work. Use the supplied Aurelia data and fictional policy values.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import json
import operator
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, TypedDict
from uuid import UUID, uuid4

from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command, interrupt

class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for batch in response.generations:
                for generation in batch:
                    usage = generation.message.usage_metadata or {}
                    self.input_tokens += usage.get("input_tokens", 0)
                    self.output_tokens += usage.get("output_tokens", 0)

counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=650, max_retries=0)
started = time.perf_counter()
DB = Path(data("capstone/service.db")).resolve()
LIMIT = {"recursion_limit": 24}

def part_facts(part_no):
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        db.row_factory = sqlite3.Row
        row = db.execute("SELECT part_no, name, stock, price_eur, depot FROM parts WHERE part_no = ?",
                         (part_no,)).fetchone()
    if row is None:
        raise ValueError("Unknown Aurelia part")
    return dict(row)

def check(label, condition):
    print("check :", label, "PASS" if condition else "FAIL")
    assert condition, label

def finish_metrics(slug, server_usage=None):
    rows = server_usage or []
    result = {"model": MODEL, "rps": float(os.environ.get("COURSE_RPS", "2")),
              "calls": counter.calls + len(rows),
              "input_tokens": counter.input_tokens + sum(r.get("input_tokens", 0) for r in rows),
              "output_tokens": counter.output_tokens + sum(r.get("output_tokens", 0) for r in rows),
              "seconds": round(time.perf_counter() - started, 1)}
    print("metrics :", json.dumps(result))
    Path(slug + "-metrics.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")

## Step 1 : Private policies

Complete the owner and scope checks before returning policy data. Opening prices, ceiling and floor are fictional lab inputs. The order facts come from the read-only Aurelia database.

In [3]:
from dataclasses import dataclass
from pydantic import BaseModel, Field

FACTS = part_facts("MR-HF-01")
ORDER = {"part_no": FACTS["part_no"], "quantity": 3, "list_price_eur": FACTS["price_eur"],
         "stock": FACTS["stock"], "depot": FACTS["depot"]}

@dataclass(frozen=True)
class Principal:
    role: str
    scope: str
    policy: dict

BUYER = Principal("buyer", "purchase-quote", {"opening_eur": 210, "ceiling_eur": 280})
SELLER = Principal("seller", "sales-quote", {"opening_eur": 300, "floor_eur": 240})

def read_policy(principal, owner):
    expected_scope = {"buyer": "purchase-quote", "seller": "sales-quote"}
    if principal.role != owner or principal.scope != expected_scope[owner]:
        raise PermissionError("Policy outside this principal's scope")
    return dict(principal.policy)

@tool
def buyer_policy(runtime: ToolRuntime[Principal]) -> str:
    """Read the buyer's own opening offer and price ceiling."""
    return json.dumps(read_policy(runtime.context, "buyer"))

@tool
def seller_policy(runtime: ToolRuntime[Principal]) -> str:
    """Read the seller's own opening ask and floor price."""
    return json.dumps(read_policy(runtime.context, "seller"))

class Quote(BaseModel):
    unit_eur: int = Field(ge=1)
    reason: str

print("order :", ORDER)
print("buyer tool schema :", buyer_policy.args)
print("seller tool schema :", seller_policy.args)

order : {'part_no': 'MR-HF-01', 'quantity': 3, 'list_price_eur': 310.0, 'stock': 9, 'depot': 'Venlo'}
buyer tool schema : {}
seller tool schema : {}


In [4]:
check("buyer policy available", read_policy(BUYER, "buyer")["ceiling_eur"] == 280)
check("seller policy available", read_policy(SELLER, "seller")["floor_eur"] == 240)

check : buyer policy available PASS
check : seller policy available PASS


## Step 2 : Scoped agents

Fill the two tool lists. Each fresh proposal receives its own principal and the public order. Agent transcripts stay in the proposal helper ; only the selected price returns to the negotiation.

In [5]:
buyer = create_agent(llm, tools=[buyer_policy], context_schema=Principal, response_format=Quote,
    system_prompt="You are Aurelia procurement. Minimise the unit price for the fixed order. "
                  "Read buyer_policy before quoting. Never exceed your own ceiling. "
                  "In round 1 use your opening offer. In round 2 move halfway from your round-1 offer "
                  "toward the seller's public round-1 ask, and never above your ceiling. "
                  "Return a Quote. Other agents' private policies are unavailable.")
seller = create_agent(llm, tools=[seller_policy], context_schema=Principal, response_format=Quote,
    system_prompt="You sell Aurelia spare parts. Maximise the unit price for the fixed order. "
                  "Read seller_policy before quoting. Never quote below your own floor. "
                  "In round 1 use your opening ask. In round 2 move halfway from your round-1 ask "
                  "toward the buyer's public round-1 offer, and never below your floor. "
                  "Return a Quote. Keep the floor price out of your reason.")

def propose(agent, principal, round_no, previous):
    task = {"order": ORDER, "round": round_no, "public_previous_offers": previous}
    out = agent.invoke({"messages": [HumanMessage(json.dumps(task))]}, LIMIT, context=principal)
    quote = out["structured_response"]
    tool_names = [m.name for m in out["messages"] if isinstance(m, ToolMessage)]
    expected = principal.role + "_policy"
    check(principal.role + " read only its own tool", expected in tool_names and set(tool_names) == {expected})
    if principal.role == "buyer":
        check("buyer ceiling", quote.unit_eur <= principal.policy["ceiling_eur"])
    else:
        check("seller floor", quote.unit_eur >= principal.policy["floor_eur"])
    public = {"role": principal.role, "unit_eur": quote.unit_eur}
    print("public quote :", public)
    return public

check("private fields absent from public order", "floor_eur" not in ORDER and "ceiling_eur" not in ORDER)

check : private fields absent from public order PASS


## Step 3 : Settlement

Write the body of settle and fill the join. Run the four fixture checks before making model calls.

In [6]:
def settle(proposals, quantity, stock):
    offers = {p["role"]: p["unit_eur"] for p in proposals}
    if not 0 < quantity <= stock:
        return {"status": "review", "reason": "quantity exceeds available stock", "external_action": False}
    if offers["buyer"] < offers["seller"]:
        return {"status": "review", "reason": "price ranges do not overlap", "external_action": False}
    unit_eur = (offers["buyer"] + offers["seller"]) // 2  # midpoint, rounded down to a whole euro
    return {"status": "draft", "unit_eur": unit_eur, "quantity": quantity,
            "total_eur": unit_eur * quantity, "external_action": False}

class Negotiation(TypedDict):
    round_no: int
    previous: list[dict]
    proposals: Annotated[list[dict], operator.add]
    settlement: dict

def buyer_node(state):
    return {"proposals": [propose(buyer, BUYER, state["round_no"], state["previous"])]}

def seller_node(state):
    return {"proposals": [propose(seller, SELLER, state["round_no"], state["previous"])]}

def settle_node(state):
    return {"settlement": settle(state["proposals"], ORDER["quantity"], ORDER["stock"])}

b = StateGraph(Negotiation)
b.add_node("buyer", buyer_node)
b.add_node("seller", seller_node)
b.add_node("settle", settle_node)
b.add_edge(START, "buyer")
b.add_edge(START, "seller")
b.add_edge(["buyer", "seller"], "settle")
b.add_edge("settle", END)
negotiation = b.compile()
fixed = [{"role": "buyer", "unit_eur": 280}, {"role": "seller", "unit_eur": 240}]
check("midpoint draft", settle(fixed, 3, 9)["total_eur"] == 780)
check("rounded midpoint", settle([{"role": "buyer", "unit_eur": 261}, fixed[1]], 3, 9)["unit_eur"] == 250)
check("non-overlapping prices need review", settle([fixed[0], {"role": "seller", "unit_eur": 300}], 3, 9)["status"] == "review")
check("stock violation needs review", settle(fixed, 10, 9)["status"] == "review")

check : midpoint draft PASS
check : rounded midpoint PASS
check : non-overlapping prices need review PASS
check : stock violation needs review PASS


## Step 4 : Negotiation

Run the two bounded rounds. In round 2 each agent moves halfway toward the other's public round-1 price. A quote equal to a private limit would disclose that limit, so neither prompt asks for one. The model may keep the prices apart ; the same rule sends that result to review. Inspect the printed offers before treating a draft as acceptable. The prompts fix round 2 by design : each side moves halfway, so both quote the midpoint of the opening prices, and the private limits only bound it.

In [7]:
previous = []
rounds = []
for round_no in [1, 2]:
    out = negotiation.invoke({"round_no": round_no, "previous": previous, "proposals": []}, LIMIT)
    previous = sorted(out["proposals"], key=lambda p: p["role"])
    rounds.append(out)
    print("round :", round_no, "settlement :", out["settlement"])
    check("both proposals retained", [p["role"] for p in previous] == ["buyer", "seller"])
    check("public offers contain no private policy", all(set(p) == {"role", "unit_eur"} for p in previous))
check("opening offers conflict", rounds[0]["settlement"]["status"] == "review")
final = rounds[-1]["settlement"]
check("bounded negotiation completed", len(rounds) == 2 and not final["external_action"])
if final["status"] == "draft":
    check("draft within both policies", SELLER.policy["floor_eur"] <= final["unit_eur"] <= BUYER.policy["ceiling_eur"])
else:
    check("unresolved conflict reaches review", final["status"] == "review")
check("database unchanged", part_facts(ORDER["part_no"]) == FACTS)

check : buyer read only its own tool PASS
check : buyer ceiling PASS
public quote : {'role': 'buyer', 'unit_eur': 210}


check : seller read only its own tool PASS
check : seller floor PASS
public quote : {'role': 'seller', 'unit_eur': 300}
round : 1 settlement : {'status': 'review', 'reason': 'price ranges do not overlap', 'external_action': False}
check : both proposals retained PASS
check : public offers contain no private policy PASS


check : buyer read only its own tool PASS
check : buyer ceiling PASS
public quote : {'role': 'buyer', 'unit_eur': 255}


check : seller read only its own tool PASS
check : seller floor PASS
public quote : {'role': 'seller', 'unit_eur': 255}
round : 2 settlement : {'status': 'draft', 'unit_eur': 255, 'quantity': 3, 'total_eur': 765, 'external_action': False}
check : both proposals retained PASS
check : public offers contain no private policy PASS
check : opening offers conflict PASS
check : bounded negotiation completed PASS
check : draft within both policies PASS
check : database unchanged PASS


## Step 5 : Access denial

Use the buyer identity for the prohibited read. The lab enforces tool and data access inside one Python process. Production credentials and downstream permissions require separate service identities. The notebook author can inspect Python memory.

In [8]:
try:
    read_policy(BUYER, "seller")
    denied = False
except PermissionError as exc:
    denied = True
    print("buyer denied :", str(exc))
check("buyer cannot read seller floor", denied)
forged = Principal("seller", "purchase-quote", BUYER.policy)
try:
    read_policy(forged, "seller")
    wrong_scope_denied = False
except PermissionError:
    wrong_scope_denied = True
check("role name alone grants no access", wrong_scope_denied)
check("identity absent from model arguments", buyer_policy.args == seller_policy.args == {})

buyer denied : Policy outside this principal's scope
check : buyer cannot read seller floor PASS
check : role name alone grants no access PASS
check : identity absent from model arguments PASS


### If time permits

- Raise the seller floor above the buyer ceiling and keep the result in review
- Replace the midpoint rule with a model judge, then check its output against both policies
- Add a delivery deadline to each private policy and share only the proposed delivery date

In [9]:
finish_metrics("exercise06-buyer-and-seller")

metrics : {"model": "openai/gpt-6-luna", "rps": 0.4, "calls": 8, "input_tokens": 2054, "output_tokens": 210, "seconds": 21.4}
